In [ ]:
### Imports

import pathlib

import h5py
import matplotlib.pyplot as plt
import numpy as np
import polars as pls
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

import usv_playpen.analyses.neuronal_coactivity_engine as engine

In [ ]:
### Load the data

data_directory = pathlib.Path('/mnt/falkner/Bartul/Data/20250919_152924')

# Load tracking data
tracking_file = next(item for item in data_directory.glob('**/*_translated_rotated_metric.h5'))

with h5py.File(name=tracking_file, mode='r') as track_file:
    mouse_tracks = np.array(track_file['tracks'])
    mouse_track_names = [item.decode('utf-8') for item in list(track_file['track_names'])]
    recording_frame_rate = float(track_file['recording_frame_rate'][()])

# Load vocalization data and filter for male USVs
usv_summary_file = next(item for item in data_directory.glob('**/*_usv_summary.csv'))
usv_summary_data = pls.read_csv(usv_summary_file)
usv_summary_data = usv_summary_data.filter(pls.col('emitter') == mouse_track_names[0])

# Load spike data (in seconds, not tracking frames)
neural_data_dict = {}
for spike_file in data_directory.glob('**/*_good.npy'):
    neural_data_dict[spike_file.stem] = np.load(spike_file)[0, :]

In [ ]:
### Compute Coactivity Metrics for Complex vs. Simple Vocalizations

total_duration = mouse_tracks.shape[0] / recording_frame_rate

# Filter groups
complex_df = usv_summary_data.filter(pls.col('usv_supercategory').is_in([1, 2]))
simple_df = usv_summary_data.filter(pls.col('usv_supercategory') == 5)

# Set a time window comparable for simple and complex calls (e.g., 30 ms)
WINDOW_S = 0.030 

# Extract count matrices (neurons x trials)
complex_counts = engine.extract_snippet_matrix(
    onsets=complex_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

simple_counts = engine.extract_snippet_matrix(
    onsets=simple_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

# Compute observed metrics
complex_obs = engine.compute_coactivity_metrics(complex_counts)
simple_obs = engine.compute_coactivity_metrics(simple_counts)

# Statistical validation
# A. Bootstrap simple calls to match the sample size of complex calls (n=71)
bootstrap_results = engine.get_bootstrapped_simple_calls(
    simple_counts=simple_counts, 
    n_target=complex_counts.shape[1], 
    n_iterations=1000
)

# B. Circular temporal shuffle (1,000 shuffles per neuron, 20-60s jitter)
complex_shuffle = engine.perform_circular_shuffle(
    onsets=complex_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    total_duration=total_duration, 
    window_s=WINDOW_S, 
    min_shift_s=20.0,
    max_shift_s=60.0,
    n_shuffles=1000
)

# 7. Display Summary Results
print(f"Analysis Window: {WINDOW_S*1000:.0f}ms | Session Duration: {total_duration:.2f}s")
print("-" * 60)
print(f"COMPLEX (n={complex_counts.shape[1]}):")
print(f"  Observed r_sc: {complex_obs['r_sc']:.4f}")
print(f"  Observed Sim:  {complex_obs['similarity']:.4f}")
print(f"  Shuffled r_sc (Mean): {np.mean(complex_shuffle['r_sc']):.4f}")
print("-" * 60)
print(f"SIMPLE (n={simple_counts.shape[1]}):")
print(f"  Observed r_sc: {simple_obs['r_sc']:.4f}")
print(f"  Observed Sim:  {simple_obs['similarity']:.4f}")
print(f"  Bootstrap r_sc (Mean matched to n={complex_counts.shape[1]}): {np.mean(bootstrap_results['r_sc']):.4f}")